The 5 Mathematical Steps of PCA
1. Standardize the Data: Center the features by subtracting their means. (Optionally, divide by the standard deviation if features have vastly different scales).
2. Compute Covariance Matrix: Calculate the relationship matrix between features.
3. Eigendecomposition: Extract eigenvalues (variance magnitude) and eigenvectors (component directions) from the covariance matrix.
4. Sort and Select: Sort the eigenvalues in descending order and keep the top k components.
5. Project Data: Multiply the original centered data by the chosen eigenvectors to transition into the lower-dimensional subspace.

In [ ]:
import numpy as np

In [ ]:
class PCAFromScratch:
  def __init__(self, n_components, scale=True):
    self.n_components = n_components
    self.scale = scale
    self.components = None
    self.mean = None
    self.std = None
    self.eignvalues = None
    self.explained_variance_ratio_ = None

  def fit(self,X):
     # 1. mean centering
    self.mean = np.mean(X,axis=0)
    X_processed = X - self.mean

    if self.scale:
      self.std = np.std(X, axis=0)
      self.std[self.std == 0] == 1e-8
      X_processed = X_processed / self.std


    # 2. Compute Covariance Matrix (row = features, col = observations)
    # We divide by (n_samples - 1) which matches np.cov's default behavior.
    cov_matrix = np.cov(X_processed,rowvar=False)

    # 3. Compute Eigenvalues and Eigenvectors
    # eigh is optimized for symmetric matrices like covariance matrices
    eignvalues, eignvactors = np.linalg.eigh(cov_matrix)

    # 4. Sort eigenvalues and eigenvectors in descending order
    # eigh returns them sorted ascending, so we reverse them.
    idxs = np.argsort(eignvalues)[::-1]
    self.eignvalues = eignvalues[idxs]
    eignvactors = eignvactors[:,idxs]

    # Calculate explained variance ratio

    total_variance = np.sum(self.eignvalues)
    self.explained_variance_ratio_ = (self.eignvalues / total_variance)[:self.n_components]

    # Store the top k components (transposed so rows represent the new axes)
    self.components = eignvactors[:,:self.n_components].T

    return self

  def transform(self,X):
    # 5. Project the data onto the new principal components
    X_processed = X - self.mean
    return np.dot(X_processed, self.components.T)

  def fit_transform(self, X):
    self.fit(X)
    return self.transform(X)

In [ ]:
# Verification code (scikit-learn)
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA as SklearnPCA

#load sample data
data = load_iris()
X = data.data

# 1. run custom PCA
custom_pca = PCAFromScratch(n_components=2, scale = False)
X_custom = custom_pca.fit_transform(X)

# 2. run Scikit-learn PCA
sklearn_pca = SklearnPCA(n_components=2)
X_sklearn = sklearn_pca.fit_transform(X)

# Check if both projections align mathematically
# Note: Signs of components can be flipped (+/-) depending on eigendecomposition solvers;
# checking the absolute value guarantees correctness.

is_equivalent = np.allclose(np.abs(X_custom), np.abs(X_sklearn))

print(f"Explained Variance Ratio (Custom): {custom_pca.explained_variance_ratio_}")
print(f"Explained Variance Ratio (Sklearn): {sklearn_pca.explained_variance_ratio_}")
print(f"Are the outputs mathematically equivalent? {is_equivalent}")

Explained Variance Ratio (Custom): [0.92461872 0.05306648]
Explained Variance Ratio (Sklearn): [0.92461872 0.05306648]
Are the outputs mathematically equivalent? True
